# 🤗 Finetuning Hugging Face Models

Want to use Hugging Face models with Composer? No problem. Here, we'll walk through using Composer to fine-tune a pretrained Hugging Face BERT model.

### Recommended Background

This tutorial assumes you are familiar with transformer models for NLP and with Hugging Face.

To better understand the Composer part, make sure you're comfortable with the material in our [Getting Started][getting_started] tutorial.

### Tutorial Goals and Concepts Covered

The goal of this tutorial is to demonstrate how to fine-tune a pretrained Hugging Face transformer using the Composer library!

We will focus on fine-tuning a pretrained BERT-base model on the Stanford Sentiment Treebank v2 (SST-2) dataset. After fine-tuning, the BERT model should be able to determine if a sentence has positive or negative sentiment.

Along the way, we will touch on:

* Creating our Hugging Face BERT model, tokenizer, and data loaders
* Wrapping the Hugging Face model as a `ComposerModel` for use with the Composer trainer
* Training with Composer
* Visualization examples

Let's do this 🚀

[getting_started]: https://docs.mosaicml.com/projects/composer/en/stable/examples/getting_started.html

## Install Composer

To use Hugging Face with Composer, we'll need to install Composer *with the NLP dependencies*. If you haven't already, run:

## Import Hugging Face Pretrained Model
First, we import a pretrained BERT model (specifically, BERT-base for uncased text) and its associated tokenizer from the transformers library.

Sentiment classification has two labels, so we set `num_labels=2` when creating our model.

In [ ]:
import transformers

from llmfoundry.utils.builders import (
    build_tokenizer,
)

tokenizer_name = "EleutherAI/gpt-neox-20b"
tokenizer_kwargs = {"model_max_length": 2048}
tokenizer = build_tokenizer(tokenizer_name, tokenizer_kwargs)
tokenizer.add_special_tokens({"pad_token": "[PAD]"})

## Creating Dataloaders

Next, we will download and tokenize the SST-2 datasets.

In [2]:
import datasets
import os

# from multiprocessing import cpu_count


# Create BERT tokenizer
def _tokenize_function(sample):  # noqa: ANN001, ANN202
    """Tokenize a sentence."""
    return tokenizer(
        text=sample["sentence"], padding="max_length", max_length=256, truncation=True
    )


# Tokenize SST-2
sst2_dataset = datasets.load_dataset("glue", "sst2", num_proc=os.cpu_count() - 1)  # type: ignore[reportOptionalOperand]
tokenized_sst2_dataset = sst2_dataset.map(
    _tokenize_function,
    batched=True,
    batch_size=100,
    remove_columns=[
        "idx",
        "sentence",
    ],
)

# Split dataset into train and validation sets
train_dataset = tokenized_sst2_dataset["train"]  # type: ignore[reportIndexIssue]
eval_dataset = tokenized_sst2_dataset["validation"]  # type: ignore[reportIndexIssue]

Here, we will create a PyTorch `DataLoader` for each of the datasets generated in the previous block.

In [3]:
from torch.utils.data import DataLoader

data_collator = transformers.data.data_collator.default_data_collator
train_dataloader = DataLoader(
    train_dataset,  # type: ignore[reportArgumentType]
    batch_size=16,
    shuffle=False,
    drop_last=False,
    collate_fn=data_collator,
)
eval_dataloader = DataLoader(
    eval_dataset,  # type: ignore[reportArgumentType]
    batch_size=16,
    shuffle=False,
    drop_last=False,
    collate_fn=data_collator,
)

## Convert model to `ComposerModel`

Composer uses `HuggingFaceModel` as a convenient interface for wrapping a Hugging Face model (such as the one we created above) in a `ComposerModel`. Its parameters are:

- `model`: The Hugging Face model to wrap.
- `tokenizer`: The Hugging Face tokenizer used to create the input data
- `metrics`: A list of torchmetrics to apply to the output of `eval_forward` (a `ComposerModel` method).
- `use_logits`: A boolean which, if True, flags that the model's output logits should be used to calculate validation metrics.

See the [API Reference][api] for additional details.

[api]: https://docs.mosaicml.com/projects/composer/en/stable/api_reference/generated/composer.models.HuggingFaceModel.html

In [4]:
from torchmetrics.classification import MulticlassAccuracy

from composer.metrics import CrossEntropy
from copy import deepcopy

from llmfoundry.utils.builders import (
    build_composer_model,
)
from llmfoundry.utils.config_utils import (
    process_init_device,
)

from flower_llm.models import MPTForSequenceClassification

metrics = [CrossEntropy(), MulticlassAccuracy(num_classes=2, average="micro")]
# Get model config  - 125M
model_config = {
    "name": "mpt_causal_lm",
    "init_device": "cpu",
    "d_model": 768,
    "n_heads": 12,
    "n_layers": 12,
    "expansion_ratio": 4,
    "max_seq_len": 2048,
    "vocab_size": 50368,
    "attn_config": {
        "attn_impl": "torch",  # "flash"
    },
    "output_hidden_states": True,
}
# # Get model config  - 3B
# model_config = {
#     "name": "mpt_causal_lm",
#     "init_device": "cpu",
#     "d_model": 2560,
#     "n_heads": 20,
#     "n_layers": 32,
#     "expansion_ratio": 4,
#     "max_seq_len": 2048,
#     "vocab_size": 50368,
#     "attn_config": {
#         "attn_impl": "torch",  # "flash"
#     },
#     "output_hidden_states": True,
# }
# # Get model config  - 350M
# model_config = {
#     "name": "mpt_causal_lm",
#     "init_device": "cpu",
#     "d_model": 1024,
#     "n_heads": 16,
#     "n_layers": 24,
#     "expansion_ratio": 4,
#     "max_seq_len": 2048,
#     "vocab_size": 50368,
#     "attn_config": {
#         "attn_impl": "torch",  # "flash"
#     },
#     "output_hidden_states": True,
# }
# Get model while forcing cpu to prevent any GPU allocation
model = build_composer_model(
    name=model_config["name"],
    cfg=model_config,
    tokenizer=tokenizer,
    init_context=process_init_device(model_config, None),
    master_weights_dtype=None,
)
# Package as a trainer-friendly Composer model
assert tokenizer.pad_token_id is not None
composer_model = MPTForSequenceClassification(
    model,  # type: ignore[reportArgumentType]
    train_metrics=metrics,
    eval_metrics=deepcopy(metrics),
    num_labels=2,
    hidden_size=model_config["d_model"],
    pad_token_id=tokenizer.pad_token_id,
)

## Optimizers and Learning Rate Schedulers

The last setup step is to create an optimizer and a learning rate scheduler. We will use PyTorch's AdamW optimizer and linear learning rate scheduler since these are typically used to fine-tune BERT on tasks such as SST-2.

In [5]:
from torch.optim.adamw import AdamW
from torch.optim.lr_scheduler import LinearLR

optimizer = AdamW(
    params=composer_model.parameters(),
    lr=3e-5,
    betas=(0.9, 0.98),
    eps=1e-6,
    weight_decay=3e-6,
)
linear_lr_decay = LinearLR(optimizer, start_factor=1.0, end_factor=0, total_iters=150)

## Composer Trainer

We will now specify a Composer `Trainer` object and run our training! `Trainer` has many arguments that are described in our [documentation](https://docs.mosaicml.com/projects/composer/en/stable/api_reference/generated/composer.Trainer.html#trainer), so we'll discuss only the less-obvious arguments used below:

- `max_duration` - a string specifying how long to train. This can be in terms of batches (e.g., `'10ba'` is 10 batches) or epochs (e.g., `'1ep'` is 1 epoch), [among other options][time].
- `schedulers` - a (list of) PyTorch or Composer learning rate scheduler(s) that will be composed together.
- `device` - specifies if the training will be done on CPU or GPU by using `'cpu'` or `'gpu'`, respectively. You can omit this to automatically train on GPUs if they're available and fall back to the CPU if not.
- `train_subset_num_batches` - specifies the number of training batches to use for each epoch. This is not a necessary argument but is useful for quickly testing code.
- `precision` - whether to do the training in full precision (`'fp32'`) or mixed precision (`'amp'`). Mixed precision can provide a ~2x training speedup on recent NVIDIA GPUs.
- `seed` - sets the random seed for the training run, so the results are reproducible!

[time]: https://docs.mosaicml.com/projects/composer/en/stable/trainer/time.html

In [ ]:
from typing import cast
import uuid
import torch
from omegaconf import DictConfig
from composer import Trainer
from flower_llm.server.s3_utils import load_pretrained_model_from_path
from flower_llm.conf.base_schema import S3CommConfig

# Load the model from a checkpoint
os.environ["S3_ENDPOINT_URL"] = "http://128.232.115.0:9000"
# pretrained_model_path = "/path/to/pretrained/model"
# pretrained_model_path = "/nfs-share/ls985/projects/flower_llm/flower_llm_checkpoints/fed-3B-20240702_141112/server/25/current_server_parameters.npz"
# pretrained_model_path = "/nfs-share/ls985/projects/flower_llm/flower_llm_checkpoints/fed-350M-2024505_100605/server/19/current_server_parameters.npz"
# pretrained_model_path = "s3://checkpoints/G1kgg-centB-125M-p-20240919/server/0/current_server_parameters.npz"
pretrained_model_path = (
    "s3://checkpoints/G1kgg-centB-125M-p-20240919/ep0-ba100-rank0.pt"
)

# Create Trainer Object
trainer = Trainer(
    model=composer_model,  # This is the model from the HuggingFaceModel wrapper class.
    train_dataloader=train_dataloader,
    eval_dataloader=eval_dataloader,
    max_duration="1ep",
    optimizers=optimizer,
    schedulers=[linear_lr_decay],
    device="gpu" if torch.cuda.is_available() else "cpu",
    train_subset_num_batches=150,
    precision="amp_fp16",
    seed=17,
    load_path=pretrained_model_path if "s3://" in pretrained_model_path else None,
    load_weights_only=True,
    load_strict_model_weights=False,
    is_model_finetune=True,
)
s3_comm_config = {
    "bucket_name": "checkpoints",
    "num_attempts": 3,
    "backend_kwargs": {
        "client_config": {
            "connect_timeout": 3600,
            "read_timeout": 3600,
        }
    },
}

if "s3://" not in pretrained_model_path:
    load_pretrained_model_from_path(
        trainer=trainer,
        pretrained_model_path=pretrained_model_path,
        run_uuid=str(uuid.uuid4()),
        s3_comm_config=cast(S3CommConfig, DictConfig(s3_comm_config)),
    )
# Start training
trainer.fit()

## Visualizing Results

To check the training's validation accuracy, we read the `Trainer` object `state.eval_metrics`

In [ ]:
trainer.state.eval_metrics

In [ ]:
trainer.state.eval_metric_values

Our model reaches ~86% accuracy with only 150 iterations of training!
Let's visualize a few samples from the validation set to see how our model performs.

In [ ]:
from logging import INFO
from flwr.common import log

eval_batch = next(iter(eval_dataloader))

# Move batch to gpu
eval_batch = {
    k: v.cuda() if torch.cuda.is_available() else v for k, v in eval_batch.items()
}
with torch.no_grad():
    predictions = composer_model(eval_batch)["logits"].argmax(dim=1)

# Visualize only 5 samples
predictions = predictions[:5]

label = ["negative", "positive"]
for i, prediction in enumerate(predictions):
    sentence = sst2_dataset["validation"][i]["sentence"]  # type: ignore[reportIndexIssue]
    correct_label = label[sst2_dataset["validation"][i]["label"]]  # type: ignore[reportIndexIssue]
    log(INFO, f"Sample: {sentence}")
    log(INFO, f"Label: {correct_label}")
    log(INFO, f"Prediction: {prediction}\n")

## Save Fine-Tuned Model

Finally, to save the fine-tuned model parameters we call the PyTorch `save` method and pass it the model's `state_dict`:

In [10]:
torch.save(trainer.state.model.state_dict(), "model.pt")

## What next?

You've now seen how to use the Composer `Trainer` to fine-tune a pre-trained Hugging Face BERT on a subset of the SST-2 dataset.

If you want to keep learning more, try looking through some of the documents linked throughout this tutorial to see if you can form a deeper intuition for what's going on in these examples.

In addition, please continue to explore our tutorials and examples! Here are a couple suggestions:

* Explore domain-specific pretraining of a Hugging Face model in a second Hugging Face + Composer [tutorial][tutorial].

* Explore more advanced applications of Composer like [applying image segmentation to medical images][image_segmentation_tutorial].

* Learn about callbacks and how to apply [early stopping][early_stopping_tutorial].

* Check out the [examples][examples] repo for full examples of training large language models like GPT and BERT, image segmentation models like DeepLab, and more!

[tutorial]: https://docs.mosaicml.com/projects/composer/en/stable/examples/pretrain_finetune_huggingface.html
[examples]: https://github.com/mosaicml/examples
[image_segmentation_tutorial]: https://docs.mosaicml.com/projects/composer/en/stable/examples/medical_image_segmentation.html
[early_stopping_tutorial]: https://docs.mosaicml.com/projects/composer/en/stable/examples/early_stopping.html

## Come get involved with MosaicML!

We'd love for you to get involved with the MosaicML community in any of these ways:

### [Star Composer on GitHub](https://github.com/mosaicml/composer)

Help make others aware of our work by [starring Composer on GitHub](https://github.com/mosaicml/composer).

### [Join the MosaicML Slack](https://join.slack.com/t/mosaicml-community/shared_invite/zt-w0tiddn9-WGTlRpfjcO9J5jyrMub1dg)

Head on over to the [MosaicML slack](https://join.slack.com/t/mosaicml-community/shared_invite/zt-w0tiddn9-WGTlRpfjcO9J5jyrMub1dg) to join other ML efficiency enthusiasts. Come for the paper discussions, stay for the memes!

### Contribute to Composer

Is there a bug you noticed or a feature you'd like? File an [issue](https://github.com/mosaicml/composer/issues) or make a [pull request](https://github.com/mosaicml/composer/pulls)!